In [1]:
import pandas as pd
import re
from pymystem3 import Mystem
from nltk.corpus import stopwords
import time

# Загрузим уже обработанный файл, чтобы не ждать снова
df_processed = pd.read_csv('../data/messages_processed.csv')
df_sample = df_processed.head(1000).copy() #1000 строк для быстрого теста

def preprocess_text_slow(text):
    # постоянно пересоздает объекты
    mystem = Mystem()
    russian_stopwords = set(stopwords.words("russian"))
    
    text = str(text).lower()
    text = re.sub(r'[^а-яА-ЯёЁ\s]', ' ', text)
    tokens = mystem.lemmatize(text)
    tokens = [token for token in tokens if token.strip() and token not in russian_stopwords]
    return " ".join(tokens)

# инициализируем объекты ОДИН РАЗ за пределами функции
mystem_global = Mystem()
stopwords_global = set(stopwords.words("russian"))

def preprocess_text_fast(text, mystem_instance, stopwords_set):
    text = str(text).lower()
    text = re.sub(r'[^а-яА-ЯёЁ\s]', ' ', text)
    tokens = mystem_instance.lemmatize(text)
    tokens = [token for token in tokens if token.strip() and token not in stopwords_set]
    return " ".join(tokens)


print("--- Тестируем МЕДЛЕННУЮ версию (постоянная инициализация) ---")
start_time_slow = time.time()
_ = df_sample['text'].apply(preprocess_text_slow)
end_time_slow = time.time()
print(f"Время выполнения: {end_time_slow - start_time_slow:.2f} секунд\n")


print("--- Тестируем БЫСТРУЮ версию (однократная инициализация) ---")
start_time_fast = time.time()
# передаем уже созданные объекты в функцию
df_sample['processed_fast'] = df_sample['text'].apply(
    lambda text: preprocess_text_fast(text, mystem_global, stopwords_global)
)
end_time_fast = time.time()
print(f"Время выполнения: {end_time_fast - start_time_fast:.2f} секунд")

print(f"\nУскорение примерно в {int((end_time_slow - start_time_slow) / (end_time_fast - start_time_fast))} раз!")

--- Тестируем МЕДЛЕННУЮ версию (постоянная инициализация) ---
Время выполнения: 5241.47 секунд

--- Тестируем БЫСТРУЮ версию (однократная инициализация) ---
Время выполнения: 5106.20 секунд

Ускорение примерно в 1 раз!


In [1]:
import pandas as pd
import re
from pymystem3 import Mystem
from nltk.corpus import stopwords
import time
import nltk
from tqdm.auto import tqdm

print("Подготовка...")
tqdm.pandas()

# Загружаем данные
try:
    df = pd.read_csv('../data/messages_processed.csv').head(1000)
    print(f"Загружен датасет из {len(df)} строк для теста.")
except FileNotFoundError:
    print("Файл не найден, создаю тестовый DataFrame.")
    df = pd.DataFrame({'text': ['Мама мыла раму.', 'Тестовое сообщение для проверки скорости.'] * 2500})


# Инициализируем объекты один раз
nltk.download('stopwords', quiet=True)
russian_stopwords = set(stopwords.words("russian"))
mystem = Mystem()
print("Подготовка завершена.")

def preprocess_texts_mystem_batch(texts_list: list, mystem_instance: Mystem, stopwords_set: set) -> list:
    """
    Принимает список текстов и обрабатывает их одной пачкой.
    """
    # Базовая очистка (делаем в цикле, это быстро)
    cleaned_texts = [re.sub(r'[^а-яА-ЯёЁ\s]', ' ', str(text).lower()) for text in texts_list]
    
    # Соединяем все тексты в одну гигантскую строку через уникальный разделитель
    # Разделитель не должен содержать кириллицу
    separator = ' |!| '
    full_text = separator.join(cleaned_texts)
    
    # Лемматизируем всё за один вызов Mystem (самая долгая часть)
    all_lemmas = mystem.lemmatize(full_text)
    
    # Собираем леммы обратно в строку и разрезаем по разделителю
    lemmatized_full_text = "".join(all_lemmas)
    lemmatized_texts = lemmatized_full_text.split(separator)
    
    # Финальная очистка от стоп-слов и лишних пробелов
    final_texts = []
    for text in lemmatized_texts:
        tokens = text.split()
        clean_tokens = [word for word in tokens if word not in stopwords_set]
        final_texts.append(" ".join(clean_tokens))
        
    return final_texts


batch_size = 1000  # Можно подобрать оптимальный размер, 500-2000 обычно хорошо
results = []

print(f"\n--- Начинаю пакетную обработку с Mystem (размер батча: {batch_size}) ---")
start_time_batch = time.time()

for i in tqdm(range(0, len(df), batch_size)):
    batch_texts = df['text'][i:i + batch_size].tolist()
    processed_batch = preprocess_texts_mystem_batch(batch_texts, mystem, russian_stopwords)
    results.extend(processed_batch)

df['processed_mystem_batch'] = results

end_time_batch = time.time()
processing_time = end_time_batch - start_time_batch

print(f"\nВремя пакетной обработки: {processing_time:.2f} секунд")

old_time_estimated = 5241.47 * (len(df) / 1000)
print(f"Примерное старое время: ~{old_time_estimated:.2f} секунд")
if processing_time > 0:
    print(f"Ожидаемое ускорение примерно в {int(old_time_estimated / processing_time)} раз!")

display(df.head())

Подготовка...
Загружен датасет из 1000 строк для теста.
Подготовка завершена.

--- Начинаю пакетную обработку с Mystem (размер батча: 1000) ---


  0%|          | 0/1 [00:00<?, ?it/s]


Время пакетной обработки: 4128.86 секунд
Примерное старое время: ~5241.47 секунд
Ожидаемое ускорение примерно в 1 раз!


,id,text,text_len,processed_text,processed_mystem_batch
0,18789fab-71ae-4b8b-9a65-c653af3a1d94,Авария 10001\nИС: Keycloak IdP\nОписание: 1018...,122,авария иса описание назначать группа платежный...,авария иса описание назначать группа платежный...
1,7210f58d-88ba-491b-812b-9d4d150dcb65,Авария 10002\nИС: GitLab CI\nОписание: 3500\nН...,127,авария иса описание назначать группа операцион...,авария иса описание назначать группа операцион...
2,727d04ef-8e6c-437a-8685-69d25c0f0f97,Авария 10003\nИС: Deposits Back-Office\nОписан...,185,авария иса описание недоступность авторизация ...,авария иса описание недоступность авторизация ...
3,acdaea40-2637-4055-998d-026ad2f21148,Авария 10004\nИС: ElasticSearch\nОписание: 101...,121,авария иса описание назначать группа,авария иса описание назначать группа
4,a6b7f5c4-cf67-47da-b0e2-7c9c604d01b6,Авария 10005\nИС: Monitoring (Zabbix)\nОписани...,170,авария иса описание высокий загрузка узел назн...,авария иса описание высокий загрузка узел назн...


In [4]:
import pandas as pd
import re
from pymorphy3 import MorphAnalyzer
from nltk.corpus import stopwords
import time
import nltk
from tqdm.auto import tqdm

print("Подготовка...")
tqdm.pandas(desc="Обработка текста")

# Используем тот же DataFrame, что и в прошлом примере
df = pd.read_csv('../data/messages_processed.csv').head(1000)

# Инициализируем объекты один раз
nltk.download('stopwords', quiet=True)
russian_stopwords = set(stopwords.words("russian"))
morph = MorphAnalyzer() # Создаем объект Pymorphy3
print("Подготовка завершена.")


def preprocess_text_pymorphy(text: str, morph_analyzer: MorphAnalyzer, stopwords_set: set) -> str:
    """
    Быстрая обработка текста с использованием Pymorphy3.
    """
    text = str(text).lower()
    text = re.sub(r'[^а-яА-ЯёЁ\s]', ' ', text)
    
    tokens = text.split()
    lemmatized_tokens = [
        morph_analyzer.parse(word)[0].normal_form
        for word in tokens
        if word not in stopwords_set
    ]
    
    return " ".join(lemmatized_tokens)

# --- 3. Запуск обработки ---
print(f"\n--- Начинаю быструю обработку с Pymorphy3 ---")
start_time_pymorphy = time.time()

# Используем .progress_apply() для отображения прогресс-бара
df['processed_pymorphy'] = df['text'].progress_apply(
    lambda text: preprocess_text_pymorphy(text, morph, russian_stopwords)
)

end_time_pymorphy = time.time()
processing_time_pymorphy = end_time_pymorphy - start_time_pymorphy

print(f"\nВремя обработки с Pymorphy3: {processing_time_pymorphy:.2f} секунд")

# Сравнение
old_time_estimated = 5241.47 * (len(df) / 1000)
print(f"Примерное старое время: ~{old_time_estimated:.2f} секунд")
if processing_time_pymorphy > 0:
    print(f"Ожидаемое ускорение примерно в {int(old_time_estimated / processing_time_pymorphy)} раз!")

display(df.head())

Подготовка...
Подготовка завершена.

--- Начинаю быструю обработку с Pymorphy3 ---


Обработка текста:   0%|          | 0/1000 [00:00<?, ?it/s]


Время обработки с Pymorphy3: 0.60 секунд
Примерное старое время: ~5241.47 секунд
Ожидаемое ускорение примерно в 8807 раз!


,id,text,text_len,processed_text,processed_pymorphy
0,18789fab-71ae-4b8b-9a65-c653af3a1d94,Авария 10001\nИС: Keycloak IdP\nОписание: 1018...,122,авария иса описание назначать группа платежный...,авария иса описание назначить группа платёжный...
1,7210f58d-88ba-491b-812b-9d4d150dcb65,Авария 10002\nИС: GitLab CI\nОписание: 3500\nН...,127,авария иса описание назначать группа операцион...,авария иса описание назначить группа операцион...
2,727d04ef-8e6c-437a-8685-69d25c0f0f97,Авария 10003\nИС: Deposits Back-Office\nОписан...,185,авария иса описание недоступность авторизация ...,авария иса описание недоступность авторизация ...
3,acdaea40-2637-4055-998d-026ad2f21148,Авария 10004\nИС: ElasticSearch\nОписание: 101...,121,авария иса описание назначать группа,авария иса описание назначить группа
4,a6b7f5c4-cf67-47da-b0e2-7c9c604d01b6,Авария 10005\nИС: Monitoring (Zabbix)\nОписани...,170,авария иса описание высокий загрузка узел назн...,авария иса описание высокий загрузка узел назн...


In [5]:

df = pd.read_csv('../data/messages_processed.csv')


print(f"\n--- Начинаю быструю обработку всех текстовых данных с Pymorphy3 ---")
start_time_pymorphy = time.time()

# Используем .progress_apply() для отображения прогресс-бара
df['processed_pymorphy'] = df['text'].progress_apply(
    lambda text: preprocess_text_pymorphy(text, morph, russian_stopwords)
)

end_time_pymorphy = time.time()
processing_time_pymorphy = end_time_pymorphy - start_time_pymorphy

print(f"\nВремя обработки с Pymorphy3: {processing_time_pymorphy:.2f} секунд")

# Сравнение обработки всех результатов
old_time_estimated = 84240
print(f"Примерное старое время: ~{old_time_estimated:.2f} секунд")
if processing_time_pymorphy > 0:
    print(f"Ожидаемое ускорение примерно в {int(old_time_estimated / processing_time_pymorphy)} раз!")

display(df.head())


--- Начинаю быструю обработку всех текстовых данных с Pymorphy3 ---


Обработка текста:   0%|          | 0/45842 [00:00<?, ?it/s]


Время обработки с Pymorphy3: 47.47 секунд
Примерное старое время: ~84240.00 секунд
Ожидаемое ускорение примерно в 1774 раз!


,id,text,text_len,processed_text,processed_pymorphy
0,18789fab-71ae-4b8b-9a65-c653af3a1d94,Авария 10001\nИС: Keycloak IdP\nОписание: 1018...,122,авария иса описание назначать группа платежный...,авария иса описание назначить группа платёжный...
1,7210f58d-88ba-491b-812b-9d4d150dcb65,Авария 10002\nИС: GitLab CI\nОписание: 3500\nН...,127,авария иса описание назначать группа операцион...,авария иса описание назначить группа операцион...
2,727d04ef-8e6c-437a-8685-69d25c0f0f97,Авария 10003\nИС: Deposits Back-Office\nОписан...,185,авария иса описание недоступность авторизация ...,авария иса описание недоступность авторизация ...
3,acdaea40-2637-4055-998d-026ad2f21148,Авария 10004\nИС: ElasticSearch\nОписание: 101...,121,авария иса описание назначать группа,авария иса описание назначить группа
4,a6b7f5c4-cf67-47da-b0e2-7c9c604d01b6,Авария 10005\nИС: Monitoring (Zabbix)\nОписани...,170,авария иса описание высокий загрузка узел назн...,авария иса описание высокий загрузка узел назн...


In [7]:
# --- Тестирование функции предобработки ---

test_messages = [
    "Срочно! система не смогла обнаружить сервер!",
    "Внимание, наблюдается высокая загрузка CPU на узле monitoring-db-01.",
    "Плановые технические работы на системе GitLab CI с 22:00 до 23:00.",
    "Ошибка авторизации пользователя в ИС Keycloak IdP.",
    "Недоступность Deposits Back-Office, разбираемся.",
    "Завтра в 10 утра будет перезагрузка.",
    "Проблема с платежным шлюзом, транзакции не проходят."
]

print("--- Результаты тестирования функции очистки ---")
for message in test_messages:
    processed = preprocess_text_pymorphy(message, morph, russian_stopwords) 
    print(f"Исходное:  {message}")
    print(f"Обработанное: {processed}\n")

--- Результаты тестирования функции очистки ---
Исходное:  Срочно! система не смогла обнаружить сервер!
Обработанное: срочно система смочь обнаружить сервер

Исходное:  Внимание, наблюдается высокая загрузка CPU на узле monitoring-db-01.
Обработанное: внимание наблюдаться высокий загрузка узел

Исходное:  Плановые технические работы на системе GitLab CI с 22:00 до 23:00.
Обработанное: плановый технический работа система

Исходное:  Ошибка авторизации пользователя в ИС Keycloak IdP.
Обработанное: ошибка авторизация пользователь иса

Исходное:  Недоступность Deposits Back-Office, разбираемся.
Обработанное: недоступность разбираться

Исходное:  Завтра в 10 утра будет перезагрузка.
Обработанное: завтра утро перезагрузка

Исходное:  Проблема с платежным шлюзом, транзакции не проходят.
Обработанное: проблема платёжный шлюз транзакция проходить

